# QLoRA fine-tune: Qwen3-4B -> APUSH item writer (v2)

Distills the strict-clean, answer-balanced frontier-teacher dataset (`data/generated/train_sft_clean.jsonl`) into Qwen3-4B, training **only on the JSON assistant response** while masking the prompt.

**Runtime:** Colab GPU. A free **T4** is enough for this 4B QLoRA run. Set `Runtime -> Change runtime type -> GPU`.

**Data:** clones the repo to read `train_sft_clean.jsonl`. If the repo is private, use the upload fallback in the dataset cell.

**Run naming:** checkpoints, local adapters, GGUF exports, Drive folders, and Hugging Face artifacts derive from `RUN_NAME = "qwen3-4b-apush-v2-clean"`, so v2 outputs do not collide with the earlier M3/v1 run.

Pipeline: install -> load 4-bit model -> add LoRA -> load+template data -> train (loss on response) -> sanity-generate -> save adapter (+ optional GGUF for Ollama).


In [ ]:
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "CPU"
print(DEVICE_NAME, "| CUDA available:", CUDA_AVAILABLE)

if not CUDA_AVAILABLE:
    raise RuntimeError("Enable a GPU runtime before loading the 4-bit model.")


In [ ]:
# 1) Install Unsloth (pulls transformers/trl/peft/bitsandbytes). torchao is required
#    by unsloth_zoo, but only run the fallback lines below if the import/load cell fails.
%pip install -q unsloth
%pip install -q --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Diagnostics: if the load cell crashes on torchao, capture these versions.
import importlib.metadata as md
import torch

def _version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return "not installed"

print(
    "torch:", torch.__version__,
    "| torchao:", _version("torchao"),
    "| transformers:", _version("transformers"),
)

# FALLBACK ONLY if the next cell errors on torchao `_wrap_tensor_autograd`.
# Uncomment one line, then Runtime -> Restart session -> Run All.
# %pip install -q --upgrade torchao
# %pip install -q --upgrade "torch>=2.6" torchao
# %pip install -q "torchao==0.9.0"


In [ ]:
# 2) Load Qwen3-4B in 4-bit. Keep v2 run/artifact names centralized here.
from unsloth import FastLanguageModel
import torch

BASE_MODEL_NAME = "unsloth/Qwen3-4B"   # keep aligned with the exact base eval candidate; record revision/digest after run
RUN_NAME = "qwen3-4b-apush-v2-clean"
MAX_SEQ_LEN = 4096                      # prompt (~2.1k tok) + JSON completion fits comfortably
SEED = 42

HF_REPO_ID = f"rohanpalviela/{RUN_NAME}-lora"
ADAPTER_DIR = f"{RUN_NAME}-lora"
GGUF_SAVE_DIR = f"{RUN_NAME}-gguf"
REPO_URL = "https://github.com/RohanPalivela/slm.git"
REPO_DIR = "slm"

BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,      # auto (bf16 on Ampere+, fp16 on T4)
)


In [ ]:
# 3) Attach LoRA adapters (QLoRA). r=16 is a solid default for a 4B on this data size.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)


In [ ]:
# 4) Get the dataset. Option A: clone the repo (public).
import os

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

DATA = os.path.join(REPO_DIR, "data/generated/train_sft_clean.jsonl")

# Option B (private repo): comment the clone above and upload the file instead.
# from google.colab import files
# up = files.upload()
# DATA = next(iter(up))

if not os.path.exists(DATA):
    raise FileNotFoundError(f"Dataset not found: {DATA}")
print("dataset:", DATA)


In [ ]:
# 5) Apply the Qwen3 chat template to each example -> a single `text` field.
#    Qwen3 may insert an empty <think></think> wrapper even with enable_thinking=False;
#    strip that wrapper so the trained assistant span is JSON only.
from datasets import load_dataset
import re

raw_dataset = load_dataset("json", data_files=DATA, split="train")


def strip_empty_think_block(text):
    return re.sub(r"<think>\s*</think>\s*", "", text)


def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    return {"text": strip_empty_think_block(text)}


dataset = raw_dataset.map(to_text, remove_columns=raw_dataset.column_names)
print(dataset)
print("train examples:", len(dataset))
print(dataset[0]["text"][:600])


In [ ]:
# 5b) RESILIENCE: checkpoint to Google Drive so a Colab disconnect can resume.
#     Colab free T4 sessions drop periodically; local /content is wiped on reset.
import os

USE_DRIVE = True   # set False to keep checkpoints only on the ephemeral Colab disk

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = f"/content/drive/MyDrive/{RUN_NAME}-outputs"
else:
    OUTPUT_DIR = f"{RUN_NAME}-outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("checkpoints ->", OUTPUT_DIR)


In [ ]:
# 6) Trainer. train_on_responses_only masks the prompt so loss is on the JSON only.
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch 16
        warmup_steps=5,
        num_train_epochs=2,              # ~78 optimizer steps on the 629-item strict-clean set
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=SEED,
        output_dir=OUTPUT_DIR,
        save_strategy="steps",
        save_steps=20,
        save_total_limit=3,
        report_to="none",
        bf16=BF16,
        fp16=not BF16,
    ),
)

# Qwen3 uses ChatML turn markers.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


In [ ]:
# 7) Train. Auto-resume from the newest checkpoint if a prior run was cut off.
#    In a fresh runtime, re-run cells 1-6 first; this then picks up where it stopped.
import os
import re


def latest_checkpoint(directory):
    if not os.path.isdir(directory):
        return None
    checkpoints = []
    for name in os.listdir(directory):
        match = re.fullmatch(r"checkpoint-(\d+)", name)
        path = os.path.join(directory, name)
        if match and os.path.isdir(path):
            checkpoints.append((int(match.group(1)), path))
    return max(checkpoints)[1] if checkpoints else None


def strip_empty_think_block(text):
    return re.sub(r"^\s*<think>\s*</think>\s*", "", text).strip()


# Response-mask sanity check: the non-masked labels should decode to assistant JSON only.
mask_sample = trainer.train_dataset[0]
label_ids = mask_sample["labels"]
input_ids = mask_sample["input_ids"]
trained_ids = [int(tok) for tok, label in zip(input_ids, label_ids) if int(label) != -100]
trained_text = tokenizer.decode(trained_ids, skip_special_tokens=True).strip()
json_text = strip_empty_think_block(trained_text)
print("raw trained span preview:")
print(trained_text[:1000])
if json_text != trained_text:
    print("stripped empty Qwen3 think wrapper; JSON span preview:")
    print(json_text[:1000])
assert json_text.startswith("[") and '"archetype"' in json_text, "response mask is not isolating assistant JSON"

ckpt = latest_checkpoint(OUTPUT_DIR)
print(f"resuming from {ckpt}" if ckpt else "fresh run (no checkpoint found)")
stats = trainer.train(resume_from_checkpoint=ckpt)
print(stats)


In [ ]:
# 8) Sanity-generate on a held-out source (not in TRAIN) to eyeball tuned output.
import os
import sys
import torch

sys.path.insert(0, REPO_DIR)
from eval.prompt_loader import LitmusPrompt

FastLanguageModel.for_inference(model)

prompt = LitmusPrompt.from_file(os.path.join(REPO_DIR, "prompts/litmus_generation_prompt.md"))
source = "Federalist No. 10 ... the tendency to break and control the violence of faction ..."
attribution = "James Madison, Federalist No. 10, 1787"
note = "Federalist arguments for an extended republic as a response to faction."
system_prompt, user_prompt = prompt.build(
    source=source,
    attribution=attribution,
    note=note,
    n=1,
    archetypes="CAUSE_OF_SOURCE",
    difficulty="operational / test-day",
    include_fewshot=False,
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]
ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    out = model.generate(
        input_ids=ids,
        max_new_tokens=768,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))


In [ ]:
# 9) Save adapter + export GGUF, then persist artifacts so Colab cannot eat them.
#    The GGUF is what Ollama needs for base-vs-tuned eval.
import glob
import os
import shutil
from huggingface_hub import HfApi, login

SAVE_GGUF_TO_DRIVE = USE_DRIVE
UPLOAD_ADAPTER_TO_HF = True
UPLOAD_GGUF_TO_HF = True

# Save the LoRA adapter locally.
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved adapter ->", ADAPTER_DIR)

api = HfApi()
if UPLOAD_ADAPTER_TO_HF or UPLOAD_GGUF_TO_HF:
    login()
    api.create_repo(HF_REPO_ID, repo_type="model", exist_ok=True)

if UPLOAD_ADAPTER_TO_HF:
    model.push_to_hub(HF_REPO_ID, token=True)
    tokenizer.push_to_hub(HF_REPO_ID, token=True)
    print("uploaded adapter ->", HF_REPO_ID)

# Export the merged model to q4_k_m GGUF for Ollama.
export_info = model.save_pretrained_gguf(
    GGUF_SAVE_DIR,
    tokenizer,
    quantization_method="q4_k_m",
) or {}
print(export_info)

gguf_dir = export_info.get("gguf_directory") or f"{GGUF_SAVE_DIR}_gguf"
gguf_files = export_info.get("gguf_files") or sorted(glob.glob(os.path.join(gguf_dir, "*.gguf")))
modelfile = export_info.get("modelfile_location") or os.path.join(gguf_dir, "Modelfile")
if not gguf_files:
    raise FileNotFoundError(f"No GGUF files found in {gguf_dir}; export did not finish cleanly.")
print("GGUF files:", gguf_files)
print("Modelfile:", modelfile)

# Persistent copy in Google Drive. This survives Colab runtime resets.
if SAVE_GGUF_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        drive_dir = f"/content/drive/MyDrive/{RUN_NAME}-gguf"
        shutil.copytree(gguf_dir, drive_dir, dirs_exist_ok=True)
        print("copied GGUF folder to Drive:", drive_dir)
    except Exception as exc:
        print("WARNING: Drive copy failed:", repr(exc))

# Upload the Ollama-ready artifacts to Hugging Face.
if UPLOAD_GGUF_TO_HF:
    for path in [*gguf_files, modelfile]:
        if not path or not os.path.exists(path):
            print("skipping missing artifact:", path)
            continue
        path_in_repo = os.path.basename(path)
        print(f"uploading {path} -> {HF_REPO_ID}/{path_in_repo}")
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=path_in_repo,
            repo_id=HF_REPO_ID,
        )
    print("uploaded GGUF artifacts ->", HF_REPO_ID)

print("Next on your Mac:")
print(f"  python3 scripts/register_ollama_from_hf.py --repo {HF_REPO_ID} --local-dir models/qwen3-apush-v2-clean --model-name qwen3-apush-v2-clean:latest")


## Next: base-vs-tuned eval

1. Export the tuned model to GGUF (cell 9), then register it in Ollama as the v2 tuned candidate:

```bash
python3 scripts/register_ollama_from_hf.py --repo rohanpalviela/qwen3-4b-apush-v2-clean-lora --local-dir models/qwen3-apush-v2-clean --model-name qwen3-apush-v2-clean:latest
```

2. Add/verify it in `eval/models.json` as a second `candidate` alongside base `qwen3:4b`. Recommended names:
   - base: `qwen3-4b` -> `qwen3:4b`
   - tuned v2 clean: `qwen3-apush-v2-clean` -> `qwen3-apush-v2-clean:latest`
3. Grow `EVAL_HELDOUT` to ~25-30 disjoint sources, then run the harness on that split and compare **near-miss (expert-quality)** rates: tuned v2 > base is the win.

```bash
python3 eval/harness.py --split EVAL_HELDOUT --runs 3 --n 6
```
